In [30]:
import os
import pandas as pd

root_dir = 'dataset/images'
output_csv_path = 'dataset.csv'

data = []

for dirpath, dirnames, filenames in os.walk(root_dir):
    if dirpath != root_dir:
        label = os.path.basename(dirpath)
        for filename in filenames:
            file_path_full = os.path.normpath(os.path.join(dirpath, filename))
            data.append({'file_path': file_path_full, 'label': label})

df = pd.DataFrame(data)

df.to_csv(output_csv_path, index=False)

In [2]:
import pandas as pd
df = pd.read_csv("dataset.csv")

In [2]:
df.head()

,file_path,label
0,"dataset/images/Apple,Leaf Rust/plant_96067.jpg","Apple,Leaf Rust"
1,"dataset/images/Apple,Leaf Rust/plant_99354.jpg","Apple,Leaf Rust"
2,"dataset/images/Apple,Leaf Rust/plant_97379.jpg","Apple,Leaf Rust"
3,"dataset/images/Apple,Leaf Rust/plant_101192.jpg","Apple,Leaf Rust"
4,"dataset/images/Apple,Leaf Rust/plant_96701.jpg","Apple,Leaf Rust"


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 137000 entries, 0 to 137006
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   file_path  137000 non-null  object
 1   label      137000 non-null  object
dtypes: object(2)
memory usage: 3.1+ MB


In [8]:
df['label'].value_counts()

label
Tomato,Yellow Leaf Curl Virus    7967
Apple,Leaf Rust                  7554
Apple,Healthy                    6269
Tomato,Late Blight               5923
Tomato,Bacterial Spot            5784
Apple,Brown Spot                 5655
Tomato,Septoria Leaf Spot        5545
Orange,Citrus Greening           5507
Tomato,Healthy                   5358
Apple,Alternaria Blotch          5343
Soybean,Healthy                  5090
Apple,Mosaic Virus               4875
Apple,Grey Spot                  4810
Tomato,Leaf Mold                 4529
Tomato,Early Blight              4179
Tomato,Spider Mites              3858
Tomato,Target Spot               3688
Apple,Frog Eye Leaf Spot         3181
Tomato,Mosaic Virus              3163
Peach,Bacterial Spot             2297
Wheat,Leaf Rust                  1980
Pumpkin,Powdery Mildew           1835
Rice,Brown Spot                  1640
Rice,Bacterial Leaf Blight       1624
Wheat,Healthy                    1524
Rice,Blast                       1520
Bluebe

In [6]:
unique_labels = sorted(df['label'].unique())
class_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_class = {i: label for label, i in class_to_id.items()}
df['class_id'] = df['label'].map(class_to_id)

In [10]:
len(unique_labels)

60

In [11]:
df.sample(5)

,file_path,label,class_id
57867,"dataset/images/Apple,Scab/plant_114246.jpg","Apple,Scab",10
28991,"dataset/images/Tomato,Septoria Leaf Spot/plant...","Tomato,Septoria Leaf Spot",49
33056,"dataset/images/Tomato,Target Spot/plant_52451.jpg","Tomato,Target Spot",51
25414,"dataset/images/Tomato,Septoria Leaf Spot/plant...","Tomato,Septoria Leaf Spot",49
72861,"dataset/images/Wheat,Healthy/plant_132616.jpg","Wheat,Healthy",53


In [7]:
from sklearn.model_selection import train_test_split
X = df[['file_path', 'label']]
y = df['class_id']
X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=(0.15/0.90), random_state=42, stratify=y_rem)

In [13]:
X_train.shape, X_val.shape, X_test.shape

((102750, 2), (20550, 2), (13700, 2))

In [8]:
import torch
from transformers import (
    ViTForImageClassification, 
    ViTImageProcessor, 
    TrainingArguments, 
    Trainer
)
import numpy as np
import evaluate

MODEL_CHECKPOINT = 'google/vit-base-patch16-224'

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(unique_labels),
    id2label=id_to_class,
    label2id=class_to_id,
    ignore_mismatched_sizes=True 
).to(device)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([60]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([60, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="weighted")

In [10]:
training_args = TrainingArguments(
    output_dir="./vit_crop",
    num_train_epochs=5,                      
    per_device_train_batch_size=16,           
    per_device_eval_batch_size=32,            
    warmup_steps=500,                         
    weight_decay=0.01,                        
    logging_dir='./logs',
    logging_steps=250,
    save_strategy="epoch",  
    eval_strategy="epoch",                  
    load_best_model_at_end=True,              
    metric_for_best_model="f1",               
    learning_rate=2e-5,                       
    fp16=False,                               
)

In [11]:
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

In [ ]:
from datasets import Dataset
from PIL import Image
import torch

train_df = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)
val_df = pd.concat([X_val, y_val], axis=1).reset_index(drop=True)
test_df = pd.concat([X_test, y_test], axis=1).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

def preprocess_images(examples):
    images = [Image.open(path).convert("RGB") for path in examples['file_path']]
    inputs = processor(images=images, return_tensors=None)
    inputs['labels'] = examples['class_id']
    return inputs

train_dataset = train_dataset.map(
    preprocess_images, 
    batched=True, 
    batch_size=32,  # Smaller batch size
    remove_columns=['file_path', 'label', 'class_id']
)
val_dataset = val_dataset.map(
    preprocess_images, 
    batched=True, 
    batch_size=32,
    remove_columns=['file_path', 'label', 'class_id']
)
test_dataset = test_dataset.map(
    preprocess_images, 
    batched=True, 
    batch_size=32,
    remove_columns=['file_path', 'label', 'class_id']
)

train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')

Map: 100%|██████████| 13700/13700 [00:25<00:00, 537.53 examples/s]


In [ ]:
# Create a custom data collator to handle batching
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator(return_tensors="pt")

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

/Users/armaanjagirdar/Projects/ML_Project/env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 